# Исследование объявлений о продаже квартир

В вашем распоряжении данные сервиса Яндекс Недвижимость — архив объявлений о продаже квартир в Санкт-Петербурге и соседних населённых пунктах за несколько лет. Вам нужно научиться определять рыночную стоимость объектов недвижимости. Для этого проведите исследовательский анализ данных и установите параметры, влияющие на цену объектов. Это позволит построить автоматизированную систему: она отследит аномалии и мошенническую деятельность.

По каждой квартире на продажу доступны два вида данных. Первые вписаны пользователем, вторые — получены автоматически на основе картографических данных. Например, расстояние до центра, аэропорта и других объектов — эти данные автоматически получены из геосервисов. Количество парков и водоёмов также заполняется без участия пользователя.

### Откройте файл с данными и изучите общую информацию

In [1]:
import pandas as pd
import matplotlib.pyplot as plt 
import missingno as msno

ModuleNotFoundError: No module named 'missingno'

In [ ]:
data = pd.read_csv('/datasets/real_estate_data.csv', sep='\t')

In [ ]:
data.head(5)

In [ ]:
data.info()

In [ ]:
data.duplicated().sum()

Сразу видим пропуски и некорректные данные в некоторых колонках, при этом явных дубликатов в таблице нет.

In [ ]:
data.hist(bins=30, figsize=(15, 25))
plt.show()

По имеющимся сырым данным мы наблюдаем следующее:
* total_images - в основном в объявлениях было 10 фотографий;
* last_price - выводы и предполжения сделать нельзя, нужно дополнительно изучить! (это последняя цена)
* total_area - график без выбросов, но на таком общем виде можем лишь сказать приблизительную площадь, уж лучше посмотреть эти данных методом discribe();
* rooms - 2 и 3 комнатные квартиры самые популярные в наших данных;
* ceiling_height - выводы и предполжения сделать нельзя, нужно дополнительно изучить! (это высота потолков)
* floors_total - самые популярные квартиры в домах с этажность менее 10-ти этажей;
* living_area - график без выбросов, но на таком общем виде можем лишь сказать приблизительную площадь, уж лучше посмотреть эти данных методом discribe();
* floor - самые частые объявления в наших данных это квартиры на 4 этажах и ниже;
* kitchen_area - график без выбросов, но на таком общем виде можем лишь сказать приблизительную площадь, уж лучше посмотреть эти данных методом discribe();
* balcony - нужно изучить дополнительно, уж слишком много балконов 3, 4 и даже пять и вообще похоже на ошибку!
* airports_nearest - похоже на правду, сверил по яндекс.картам и действительно расстояние от 0 до 80 км не кажется странным, но глянем еще на этапе предобработки;
* cityCenters_nearest - нужно смотреть дополнительно;
* parks_around3000 - кажется что все ок, поработаем с пропусками только;
* parks_nearest - в основном до парка 500 м, выбросов нет, кажется таким данным можно верить;
* ponds_around3000 - кажется все ок, нужно подумать что делать с пропусками есть все-таки рядом парки или нет;
* ponds_nearest - 500 м самое частое расстояние до ближайщего водоема;
* days_exposition - кажется в основном объявления были опубликованы в течение полугода;

В целом мы видим где данные распределяются "нормально", а где есть аномальные значения, также эти данные могут быть задублированны. Сейчас в режиме предобработки будем все исправлять. 

### Выполните предобработку данных

In [ ]:
data.isna().sum()


In [ ]:
msno.matrix(data, labels=True)
plt.show()

In [ ]:
data['ceiling_height'].describe() 
# содержит непонятные выбросы 100 м потолок. пустоту заменим медианой

In [ ]:
data['ceiling_height'] = data['ceiling_height'].fillna(data['ceiling_height'].median())

In [ ]:
data['is_apartment'].unique() 
# раз данные пусты, будем считать что они ответ не положительный - меняем на False

In [ ]:
data['is_apartment'] = data['is_apartment'].fillna(False)
data['is_apartment'] = data['is_apartment'].astype(bool) 
#также видим некорректный тип - изменим тип данных на bool тк в описании колонок указано что это булевый тип

In [ ]:
data['balcony'] = data['balcony'].fillna(0)

In [ ]:
data['balcony'].unique()

Заменили пустые значения там, где это логически возможно, в остальной части оставим как есть и не станем на дааном этапе удальять пустые строки

* ceiling_height - высота потолков не везде была заполнена пользователем (и мы можем ее угадать тк она находится в понятном для нас диапазоне)
* floors_total - не везде была заполнена пользователем
* living_area - не везде была заполнена пользователем
* is_apartment - не везде была заполнена пользователем
* kitchen_area - не везде была заполнена пользователем
* balcony - не везде была заполнена пользователем
* locality_name - не везде была заполнена пользователем
* airports_nearest - не везде была заполнена автоматически (тех сбой или нет данных)
* cityCenters_nearest - не везде была заполнена автоматически (тех сбой или нет данных)
* parks_around3000 - не везде была заполнена автоматически (тех сбой или нет данных)
* parks_nearest - не везде была заполнена автоматически (тех сбой или нет данных)
* ponds_around3000 - не везде была заполнена автоматически (тех сбой или нет данных)
* ponds_nearest - не везде была заполнена автоматически (тех сбой или нет данных)
* days_exposition - не везде была заполнена автоматически (объявление еще не снято или все-таки тех сбой)

In [ ]:
data.head(2)

In [ ]:
data.dtypes #посмотрим какие типы данных в колонках еще нужно поправить

In [ ]:
data['first_day_exposition'] = pd.DatetimeIndex(data['first_day_exposition']) 
#колонка определена как объект, для нас это дата и время - меняем тип на дату тк время нам не потребуется

In [ ]:
data['last_price'] = data['last_price'].astype(int) 
data['floors_total'] = data[data['floors_total'].notnull()]['floors_total'].astype('int32')
data['balcony'] = data[data['balcony'].notnull()]['balcony'].astype('int32')
data['parks_around3000'] = data[data['parks_around3000'].notnull()]['parks_around3000'].astype('int32')
data['ponds_around3000'] = data[data['ponds_around3000'].notnull()]['ponds_around3000'].astype('int32')
data['days_exposition'] = data[data['days_exposition'].notnull()]['days_exposition'].astype('int32')

Изменим тип там где это возможно, в данном случае это не актуально потому что у нас маленьки датасет, но в случае если б он был в млн строк мы бы уменьшили потребление памяти и ускорили работу с данными

In [ ]:
data.columns #для удобства поиска дублей

In [ ]:
len(data['locality_name'].unique()) #до обработки 365 разных наименований

In [ ]:
data['locality_name'].sort_values().unique()
# на данном этапе нас интересуют дубли в этой колонке - будем исправлять

In [ ]:
data['locality_name'] = (
    data['locality_name']
    .str.replace('ё', 'е')
    .replace(['городской поселок', 'поселок городского типа', 'поселок городского типа', 'коттеджный поселок'], 'поселок', regex=True)
    .replace(['садоводческое некоммерческое товарищество'], 'садовое товарищество', regex=True)
)

In [ ]:
len(data['locality_name'].unique()) #теперь их стало 322

Увидили отсутсвие явных дублей, построили графики для всех колонок с числовым значениями, заметили некоторые аномальные значения (выбросы и странные распределения) по ним. 

Нашли и изучили пропущенные значения в столбцах, обозначили что могло явиться причиной пропуска значений, там где это логично мы заменили данные на приблизительное значение, там где это можно сделать только наугад мы оставили пропуски, чтобы не искажать данный при дальшних расчетах. 

Также система не корректно определила тип данных в части колонок и мы это поправили. 

Изучили уникальные значения в колонках и устранили неявные дубликаты.

### Добавьте в таблицу новые столбцы

Добавьте в таблицу новые столбцы со следующими параметрами:
* цена одного квадратного метра (нужно поделить стоимость объекта на его общую площадь, а затем округлить до двух знаков после запятой);
* день недели публикации объявления (0 — понедельник, 1 — вторник и так далее);
* месяц публикации объявления;
* год публикации объявления;
* тип этажа квартиры (значения — «‎первый», «последний», «другой»);
* расстояние до центра города в километрах (переведите из м в км и округлите до ближайших целых значений).

In [ ]:
data['price_m2'] = data['last_price'] / data['total_area'].round(2)

In [ ]:
data['weekday'] = data['first_day_exposition'].dt.weekday

In [ ]:
data['month'] = data['first_day_exposition'].dt.month

In [ ]:
data['year'] = data['first_day_exposition'].dt.year

In [ ]:
data['cityCenters_nearest_km'] = (data['cityCenters_nearest'] / 1000).round(2)

In [ ]:
data['first_floor'] = (data['floor'] == 1) *1
data['last_floor'] = (data['floor'] == data['floors_total']) *1
data['other_floor'] = ((data['last_floor'] == 0) & (data['first_floor'] == 0)) *1

In [ ]:
data.head()

Дополнительные критерии созданы и работают, данные метрики (параметры) понадобятся для последующего анализа и подготовки предположений, дополнили датафрейм критерием с указанием первого и последнего этажа, добавили информацию о стоимости метра и выразили удаленность от центра города в км с точностью до двух знаков после запятой (точки).

### Проведите исследовательский анализ данных

In [ ]:
data = data[data['kitchen_area'] < data['living_area']] 
# кухня больше жилой площади и таких строк 220 - исключаем тк кухня является жилой площадью
ka = data['kitchen_area'].plot(kind='hist', bins=100, range=(5,60), title='Площадь кухни', grid=True)
ka.set_xlabel('Площадь')
ka.set_ylabel('Количество')
data['kitchen_area'].describe()

* 3/4 квартир имеют кухню площадью 11.6 $м^2$ и более
* чаще всего квартиры имеют кухню площадью 6 или 10 $м^2$ (при измении масштаба график показывает именно такие значения)

In [ ]:
ta = data['total_area'].plot(kind='hist', bins=100, range=(10,200), title='Общая площадь', grid=True)
ta.set_xlabel('Площадь')
ta.set_ylabel('Количество')
data['total_area'].describe()

Больше всего квартир с площадью 35-45 $м^2$.
Квартир с площадью от 50-60 $м^2$ примерно на 40% меньше, чем вышеуказанных, а квартир Большей площади сильно меньше  

In [ ]:
la = data['living_area'].plot(kind='hist', bins=100, range=(10,180), title='Жилая площадь', grid=True)
la.set_xlabel('Площадь')
la.set_ylabel('Количество')
data['living_area'].describe()

* Большинство квартир имеют жилую площадь около 42 $м^2$.
* Больше всего тех квартир, которые имеют жилую площадь около 20 $м^2$.
* Примерно вдвое меньше квартир с жилой площадью около 30 $м^2$

In [ ]:
lp = data['last_price'].plot(kind='hist', bins=100,  range=(400000, 11000000),title='Цена продажи', grid=True)
lp.set_xlabel('Цена')
lp.set_ylabel('Количество')
data['last_price'].describe()

* Больше всего квартир продавались за 4 млн рублей
* основные предложения на приобретение квартир находятся в диапазоне цен от 3.5 до 6.7 млн

In [ ]:
data['rooms'].value_counts()
# количество комнат, предлагаю исключить данные с 8 и более комнатами тк их совсем мало и они, вероятно, ошибочны

In [ ]:
# data.sort_values(by='rooms', ascending=False).head(20) 
# судя по площади и цене это не ошибочные данные - оставим

In [ ]:
rooms = data['rooms'].plot(kind='hist', bins=30,  range=(1, 7),title='Кол-во комнат', grid=True)
rooms.set_xlabel('Кол-во комнат')
rooms.set_ylabel('Количество квартир')

* Больше всего продавались 2-х комнатные квартиры
* Чуть меньше было однокомнатных квартир
* на третьем месте в Топе самых часторазмещаемых квартир трехкомнатные

In [ ]:
data['ceiling_height'].describe()

Есть сомнения по максимальным значениям, вероятней всего нужно значения от 24 метров поделить на 10 и получатся корректные данные. Минимальные значения (до 2.2 метра отбросить)

In [ ]:
data['ceiling_height'] = data['ceiling_height'].where(data['ceiling_height'] < 22, data['ceiling_height']/10)

In [ ]:
ch = data['ceiling_height'].plot(kind='hist', bins=90,  range=(2.4, 4),title='Высота потолка', grid=True)
ch.set_xlabel('Высота потолка')
ch.set_ylabel('Количество квартир')

На предыдущем этапе мы 38% всех значений, которые были пустыми, заполнили медианным значением и поэтому говорить о том что теперь квартир с такой высотой потолков будет не верно, но можно сформулировать следующим образом вывод по графику: преболадающая высота потолков в квартирах из объявлений 2.5 и 2.7 м

In [ ]:
first_floor_sum = data['first_floor'].sum()
last_floor_sum = data['last_floor'].sum()
other_floor_sum = data['other_floor'].sum()

sums = [first_floor_sum, last_floor_sum, other_floor_sum]

plt.bar(['Первый этаж', 'Последний этаж', 'Другой этаж'], sums)
plt.ylabel('Количество')

Квартир, расположенных на последнем этаже больше, чем на первом. А общая доля квартир, располженных на первом или последнем этажах, составляет около трети от числа квартир, расположенных на других этажах

In [ ]:
ft = data['floors_total'].plot(kind='hist', bins=30,  range=(0, 65), grid=True)
ft.set_xlabel('Количество этаже в доме')
ft.set_ylabel('Количество квартир')
data['floors_total'].describe()

* Больше всего объявлений о продаже квартир в пятиэтажках
* Чуть меньше обхявлений о продаже квартир в 10-ти этажных домах

In [ ]:
ccn = data['cityCenters_nearest_km'].plot(kind='hist', bins=30,  range=(0, 70), grid=True)
ccn.set_xlabel('Расстояние до центра города (км)')
ccn.set_ylabel('Количество квартир')
data['cityCenters_nearest_km'].describe()

Больше всего квартир располагаются в 16 км от центра города

In [ ]:
pn = data['ponds_nearest'].plot(kind='hist', bins=30,  range=(0, 1500), grid=True)
pn.set_xlabel('Расстояние до ближайшего парка (м)')
pn.set_ylabel('Количество квартир')
data['ponds_nearest'].describe()

В основном квартиры из объявлений находятся на расстоянии не более 900 м от парка.

In [ ]:
dexpo= data['days_exposition'].plot(kind='hist', bins=50, range=(0,1000), title='Срок продажи (дни)', grid=True)
dexpo.set_xlabel('Кол-во дней')
dexpo.set_ylabel('Кол-во квартир')
data['days_exposition'].describe()

In [ ]:
#Посмотрим более детально самую основную массу объявлений
dexpo= data['days_exposition'].plot(kind='hist', bins=50, range=(0,100), title='Срок продажи (дни)', grid=True)
dexpo.set_xlabel('Кол-во дней')
dexpo.set_ylabel('Кол-во квартир')


* Чаще всего квартиру продавали за 45 и 60 дней 
* 50% квартир продавали в течение 102 дня 
* Среднее время продажи квартиры составило 185 дней

Таким образом, раз в половине случаев за 102 дня квартира продается, тогда тоже расчитываем что продадим в этот срок. Если мы не продаем ее в течение 239 дней, то считаем что это весьма долго т.к. к этому сроку остается лишь 25% нераспроданных квартир

In [ ]:
data.columns
# из списка колонок вспоминаем какие данные они содержат и выбираем те, что по нашему мннию влияют на цену

In [ ]:
# оценим зависимость цены от этажности квартиры
data[['last_price', 'first_floor', 'last_floor', 'other_floor']].corr()
#тут виднее,чем на графиках

In [ ]:
pd.plotting.scatter_matrix(data[['last_price', 'first_floor', 'last_floor', 'other_floor']], figsize=(12,12))
plt.show()

* квартиры, расположенные на первых этажах дешевле остальных
* квартиры на последних этажах чуть дороже, чем на первых
* другие этажи стоят дороже первого и последнего этажей

In [ ]:
data[['last_price', 'total_area', 'living_area', 'kitchen_area']].corr()

In [ ]:
pd.plotting.scatter_matrix(data[['last_price', 'total_area', 'living_area', 'kitchen_area']], figsize=(12,12))
plt.show()

* чем больше общая площадь (жилая площадь или площадь кухни), тем выше цена
* жилая площадь менее влияет на цену, чем площадь кухни или общая площадь

* в 2014 году цены были ниже, после цены начали расти
* в 2018 году цены были максимальными, а зате снизились до уровня 2016 года
* зимой цены максимальные, а летом минимальные
* по четвергам квартиры продавались дороже
* по воскресеньям цены были ниже

In [ ]:
pd.plotting.scatter_matrix(data[['last_price', 'weekday', 'month','year']], figsize=(12,12))
plt.show()

In [ ]:
top = data['locality_name'].value_counts().head(10)

In [ ]:
price_m = data.pivot_table(index='locality_name', values='price_m2', aggfunc='mean')
top_price_m = price_m.merge(top,  how='right', left_index=True, right_index=True)
top_price_m = top_price_m.sort_values(by='price_m2', ascending=False)
top_price_m['price_m2'] = top_price_m['price_m2'].astype(int)
top_price_m

Среди 10 мест, где больше всего объявлений:
* Санкт-Петербург имеет самую дорогую стоимость за $м^2$
* Выборг имеет самую доступную цену за $м^2$

In [ ]:
data['cityCenters_nearest_km'] = data['cityCenters_nearest_km'].round()
km = (data
        .query('locality_name == "Санкт-Петербург"')
        .groupby('cityCenters_nearest_km')['price_m2']
        .mean()
        .to_frame()
        )
km['cityCenters_nearest_km'] = km.index
#проверил-работает

In [ ]:
km.plot(x='cityCenters_nearest_km', y='price_m2', kind='line', sharex=True, figsize=(10, 5))
plt.xlabel('Расстояние до центра города (км)')
plt.ylabel('Цена за м²')
plt.show()

Дороже всего $м^2$ жилья стоит в самом центре г. Санкт-Петербурга, в трех километрах от центра цена снижается почти на треть, затем возрастает до 150 тыс. (на удаленности 7 км от центра) и постепенно снижается по мере удаленности от центра. 
На расстоянии 27-28 км от центра цены резко возрастают (видимо это элитный район) с 80 тыс. до 130 и продолжают тренд к снижению увеличивая расстояние до центра.
Судя по графику можно предположить, что самым центром считаеся радиус в 3 км. 

### Напишите общий вывод

Мы получили данные, которые судя по графикам имели аномалии, а также пропущенные значения и в процессе преобработки мы заполнили пропуски где это было возможно и устранили задублированные данные так называемые неявные дубликаты. Удалили аномальные значения, например, очень низкие потолки, а также "починили" техническую ошибку, при которй часть квартир имела потомлки 100м. Затем с помощью графиков и описательной статистики данных выявили некоторые закономерности:
* больше всего продавались однокомнатные и 2-х комнатные квартиры
* больше всего объявлений квартир из пятиэтажек или десятиэтажек
* квартиры на первом этаже стоят дешевле, чем на последнем, а в целом эти обе категории дешевле остальных квартир
* Чаще всего квартиру продавали за 45 и 60 дней
* половина квартир были проданы в течение 102 дня
* большинство квартир располагаются не дальше 900 м от парка
* по мере увеличения площади кухни, общей площади или жилой площади (но в меньшей степени) повышается и цена объекта
* в 2014 году цены были сильно ниже, чем в 2018
* зимой цены максимальные, а летом минимальные
* по четвергам квартиры продавались дороже, а по воскресеньям дешевле
* в г. Санкт-Петербурге цены за $м^2$ жилья самые высокие в рамках рассмтариваемых данных и находятся на максимуме в самом центре города и постепенно снижаются по мере удаления, но есть некторые "интересные локации", которые поднимают цену вверх

<div class="alert alert-block alert-success">
 
### Комментарий ревьюера
    
#### Успех

Итоговый вывод стал прекрасным дополнением к проекту. Работа вышла достаточно насыщенной и в выводе описана вся суть исследования. Нам точно хватит данных для составления антифрод-системы сервиса Недвижимости

<div class="alert alert-block alert-info">
 
### Итоговый Комментарий ревьюера
    
#### Успех
    
Благодарю тебя за выполнение работы. Мне понравился твой проект за структурность и последовательность. Во многих местах ты подбираешь оптимальный код и автоматизируешь свою работу, а это очень пригодится в будущем. Выделить бы хотел Предобработку данных. На мой взгляд она получилась особенно удачно. Очень насыщенный итоговый вывод вышел. Старайся такие выводы делать и в будущем. 
    
    
Однако, в проекте есть несколько замечаний, которые надо исправить:

* Убрать подсчет корреляции для некоторых параметров. 
       
Еще я оставил рекомендации. Очень надеюсь, что ты учтешь их в этом и последующих проектах.
       
Жду проект после доработки. Уверен, ты справишься.
</div>

**Чек-лист готовности проекта**

Поставьте 'x' в выполненных пунктах. Далее нажмите Shift+Enter.

- [x]  Файл с данными открыт.
- [x]  Файл с данными изучен: выведены первые строки, использован метод `info()`, построены гистограммы.
- [x]  Найдены пропущенные значения.
- [x]  Пропущенные значения заполнены там, где это возможно.
- [x]  Объяснено, какие пропущенные значения обнаружены.
- [x]  В каждом столбце установлен корректный тип данных.
- [x]  Объяснено, в каких столбцах изменён тип данных и почему.
- [x]  Устранены неявные дубликаты в названиях населённых пунктов.
- [x]  Обработаны редкие и выбивающиеся значения (аномалии).
- [x]  В таблицу добавлены новые параметры:
       – цена одного квадратного метра;
       – день публикации объявления (0 - понедельник, 1 - вторник и т. д.);
       – месяц публикации объявления;
       – год публикации объявления;
       – тип этажа квартиры (значения — «первый», «последний», «другой»);
       – расстояние до центра города в километрах.
- [x]  Изучены и описаны параметры:
        - общая площадь;
        - жилая площадь;
        - площадь кухни;
        - цена объекта;
        - количество комнат;
        - высота потолков;
        - тип этажа квартиры («первый», «последний», «другой»);
        - общее количество этажей в доме;
        - расстояние до центра города в метрах;
        - расстояние до ближайшего парка.
- [x]  Выполнено задание «Изучите, как быстро продавались квартиры (столбец `days_exposition`)»:
    - построена гистограмма;
    - рассчитаны среднее и медиана;
    - описано, сколько обычно занимает продажа и указано, какие продажи можно считать быстрыми, а какие — необычно долгими.
- [x]  Выполнено задание «Определите факторы, которые больше всего влияют на общую (полную) стоимость объекта». Построены графики, которые показывают зависимость цены от параметров:
        - общая площадь;
        - жилая площадь;
        - площадь кухни;
        - количество комнат;
        - тип этажа, на котором расположена квартира (первый, последний, другой);
        - дата размещения (день недели, месяц, год).
- [x]  Выполнено задание «Посчитайте среднюю цену одного квадратного метра в 10 населённых пунктах с наибольшим числом объявлений»:
    - выделены населённые пункты с самой высокой и низкой стоимостью квадратного метра.
- [x]  Выполнено задание «Выделите квартиры в Санкт-Петербурге с помощью столбца `locality_name` и вычислите их среднюю стоимость на разном удалении от центра»:
    -  учтён каждый километр расстояния, известны средние цены квартир в одном километре от центра, в двух и так далее;
    -  описано, как стоимость объекта зависит от расстояния до центра города;
    -  построен график изменения средней цены для каждого километра от центра Петербурга.
- [x]  На каждом этапе сделаны промежуточные выводы.
- [x]  В конце проекта сделан общий вывод.